In [1]:
# from lapy import TriaMesh
import numpy as np
from neuromodes.eigen import EigenSolver
from neuromodes.io import fetch_surf
from neuromodes.mbm2 import mbm_example_workflow

In [2]:
mesh, _ = fetch_surf(density='4k') 
solver = EigenSolver(mesh).solve(100)
# print(solver)
# print(solver.emodes)
# print(solver.evals)

In [3]:
def generate_mbm_test_data(
        emodes: np.ndarray, 
        n_subjects: int = 50, 
        n_sig_modes: int = 5, 
        effect_size: float = 3.0,
        seed: int | None = 42
):
    """
    Generates synthetic vertex maps with known significant eigenmodes 
    for a two-sample group comparison.
    
    Args:
        emodes: User-supplied eigenmodes matrix (n_vertices, n_modes).
        n_subjects: Total number of subjects (will be split into two groups).
        n_sig_modes: Number of modes to artificially make "significant".
        effect_size: The mean difference between groups for the significant modes.
        seed: Random seed for reproducibility.
        
    Returns:
        maps: (n_vertices, n_subjects) array of simulated data.
        statDesignMatrix: (n_subjects, 2) boolean array for a two-sample test.
        mass: (n_vertices, n_vertices) simulated mass matrix.
        sig_modes: Array of the indices for the ground-truth significant modes.
    """
    rng = np.random.default_rng(seed)
    n_vertices, n_modes = emodes.shape
    
    # 1. Randomly select which modes will be our ground-truth significant modes
    sig_modes = rng.choice(n_modes, size=n_sig_modes, replace=False)
    sig_modes.sort()
    
    # 2. Create the two-sample design matrix
    # Group A: First half of subjects, Group B: Second half
    group_A_idx = np.arange(n_subjects // 2)
    group_B_idx = np.arange(n_subjects // 2, n_subjects)
    
    statDesignMatrix = np.zeros((n_subjects, 2), dtype=bool)
    statDesignMatrix[group_A_idx, 0] = True
    statDesignMatrix[group_B_idx, 1] = True

    # 3. Generate mode loadings (n_modes, n_subjects)
    # Start all modes with standard gaussian noise for all subjects
    mode_loadings = rng.normal(loc=0.0, scale=1.0, size=(n_modes, n_subjects))
    
    # Inject the signal into the chosen significant modes
    # Group A gets a positive shift, Group B gets a negative shift
    mode_loadings[sig_modes[:, None], group_A_idx] = rng.normal(loc=+effect_size, scale=1.0, size=(len(sig_modes), len(group_A_idx)))
    mode_loadings[sig_modes[:, None], group_B_idx] = rng.normal(loc=-effect_size, scale=1.0, size=(len(sig_modes), len(group_B_idx)))
        
    # 4. Project the mode loadings back to vertex space
    # (n_vertices, n_modes) @ (n_modes, n_subjects) -> (n_vertices, n_subjects)
    maps = emodes @ mode_loadings
    
    # Add a layer of vertex-wise spatial noise to test the permutation test
    vertex_noise = rng.normal(loc=0.0, scale=0.5, size=(n_vertices, n_subjects))
    maps += vertex_noise

    
    return maps, statDesignMatrix, sig_modes

In [ ]:
# Assuming you have your emodes loaded: shape (10000, 100) for example
# emodes = load_your_emodes() 

maps, statDesignMatrix, ground_truth_sig_modes = generate_mbm_test_data(
    solver.emodes, 
    n_subjects=50, 
    n_sig_modes=5,
    effect_size=3.0, 
    seed=314
)

print(f"Ground truth significant modes: {ground_truth_sig_modes}")

# Run your workflow
reconMap, sig_mode_indices = mbm_example_workflow(
    maps=maps,
    emodes=solver.emodes,
    mass=solver.mass,
    statTest='two sample',
    statDesignMatrix=statDesignMatrix,
    statPThr=0.05,
    statFDR=True,
    n_modes=solver.emodes.shape[1],
    n_permutations=100,
    seed=27
)

print(f"I identified significant modes: {sig_mode_indices}")

Ground truth significant modes: [44 55 66 70 89]
(4002, 4002)
(4002,)
(100, 1, 100)
(100, 1000)


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()